In [1]:
import pandas as pd
from chembl_webresource_client.new_client import new_client

def collect_her2_data(uniprot_id="P04626"):
    print(f"Connecting to ChEMBL and looking up Target for UniProt ID: {uniprot_id}...")
    
    # 1. Look up the ChEMBL target ID using the UniProt ID
    target_api = new_client.target
    target_query = target_api.filter(target_components__accession=uniprot_id).only('target_chembl_id', 'organism')
    
    # Ensure we target the Human version (Homo sapiens) as done in the paper
    human_targets = [t for t in target_query if t['organism'] == 'Homo sapiens']
    
    if not human_targets:
        print("No human target found for this UniProt ID.")
        return None
    
    # Take the first matched human target ID
    target_chembl_id = human_targets[0]['target_chembl_id']
    print(f"Found Target ChEMBL ID: {target_chembl_id} for Homo sapiens.")
    
    # 2. Query bioactivity data for this target
    print("Fetching bioactivity records...")
    activity_api = new_client.activity
    
    # Replicating paper criteria: Filter by target and look specifically for IC50 values
    activities = activity_api.filter(
        target_chembl_id=target_chembl_id, 
        standard_type="IC50"
    ).only(
        'molecule_chembl_id', 
        'canonical_smiles', 
        'standard_type', 
        'standard_value', 
        'standard_units'
    )
    
    # 3. Convert records into a Pandas DataFrame
    df = pd.DataFrame.from_records(activities)
    print(f"Retrieved {len(df)} raw entries.")
    
    # 4. Data Cleaning & Preparation (Following the paper's steps)
    # Drop rows where critical fields are missing (null values)
    df = df.dropna(subset=['canonical_smiles', 'standard_value', 'standard_units'])
    
    # Ensure units are strictly in nM (nanomolar) to match IC50 classification standards
    df = df[df['standard_units'] == 'nM']
    
    # Convert value column to numeric
    df['standard_value'] = pd.to_numeric(df['standard_value'], errors='coerce')
    df = df.dropna(subset=['standard_value'])
    
    # Drop duplicate entries based on the molecule structure
    df = df.drop_duplicates(subset=['molecule_chembl_id'])
    df = df.drop_duplicates(subset=['canonical_smiles'])
    
    # 5. Activity Classification (<= 0.5 uM or <= 500 nM is Active, rest is Inactive)
    # 0.5 uM = 500 nM
    df['activity_class'] = df['standard_value'].apply(lambda x: 'active' if x <= 500.0 else 'inactive')
    
    print("\n--- Processing complete ---")
    print(f"Cleaned unique compounds: {len(df)}")
    print(df['activity_class'].value_counts())
    
    return df

# Run the data collection
her2_dataset = collect_her2_data()

# Optional: Save the dataset to a CSV file to use for machine learning training
if her2_dataset is not None:
    her2_dataset.to_csv("HER2_ChEMBL_bioactivity_data.csv", index=False)
    print("\nDataset successfully saved to 'HER2_ChEMBL_bioactivity_data.csv'.")

Connecting to ChEMBL and looking up Target for UniProt ID: P04626...
Found Target ChEMBL ID: CHEMBL1824 for Homo sapiens.
Fetching bioactivity records...
Retrieved 5266 raw entries.

--- Processing complete ---
Cleaned unique compounds: 3352
activity_class
active      1953
inactive    1399
Name: count, dtype: int64

Dataset successfully saved to 'HER2_ChEMBL_bioactivity_data.csv'.


In [ ]:
# import pandas as pd
# import requests

# # 1. Define the URL for cross-assay bioactivity using the ERBB2 Gene ID (2064)
# gene_id = 2064
# url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/assay/targets/gene/id/{gene_id}/doseresponse/CSV"

# print("Fetching ERBB2 bioactivity data from PubChem... This may take a moment.")
# try:
#     # 2. Pull the data into a Pandas DataFrame
#     df = pd.read_csv(url)
#     print(f"Successfully downloaded {len(df)} bioactivity data points!")
    
#     # Quick peak at what the columns look like
#     print(df[['cid', 'activity', 'acname', 'acvalue']].head())
    
#     # 3. Save raw file
#     df.to_csv("erbb2_raw_pubchem_data.csv", index=False)

# except Exception as e:
#     print(f"An error occurred: {e}")

Fetching ERBB2 bioactivity data from PubChem... This may take a moment.
An error occurred: HTTP Error 400: PUGREST.BadRequest


In [ ]:
import pandas as pd
import requests
import io
import time

# --- STEP A: TARGET BIOLOGICAL ASSAYS VIA GENE ID 596 ---
print("Fetching Assays linked to Gene ID 2046...")
gene_id = "2064"
# Requesting all assay descriptions (AIDs) targeting human HERR-2 gene 
url_assays = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/assay/target/geneid/{gene_id}/aids/TXT"
response = requests.get(url_assays)

if response.status_code != 200:
    raise ValueError("Failed to fetch Assay IDs from Pubchem. Check connection or Gene ID.")

# Convert string response to list of integer Assay IDs (AIDs)
all_aids = [int(aid) for aid in response.text.splitlines() if aid.strip()]
print(f"Found {len(all_aids)} unique assays associated with Gene ID {gene_id}.")



Fetching Assays linked to Gene ID 596...
Found 1000 unique assays associated with Gene ID 2064.


In [18]:
all_aids

[1433,
 1622,
 1904,
 1982,
 3366,
 3367,
 40020,
 40021,
 42142,
 51584,
 68095,
 68096,
 68097,
 68098,
 68099,
 68100,
 68101,
 68102,
 68103,
 68104,
 68105,
 68106,
 69712,
 69713,
 69714,
 69715,
 69716,
 69717,
 69874,
 70241,
 79614,
 79615,
 79617,
 81560,
 81561,
 81567,
 81574,
 81575,
 81579,
 81580,
 81581,
 81582,
 81583,
 81584,
 81723,
 81724,
 87018,
 91107,
 91108,
 92132,
 105058,
 144357,
 164334,
 164335,
 164336,
 164337,
 164338,
 164339,
 164340,
 215071,
 218938,
 219060,
 219061,
 226205,
 226585,
 227709,
 235147,
 235148,
 235920,
 240497,
 240554,
 240635,
 240906,
 241491,
 242383,
 244158,
 244178,
 248902,
 248914,
 248969,
 254695,
 254834,
 254920,
 254948,
 254975,
 255842,
 255868,
 255958,
 256709,
 258045,
 258046,
 259431,
 260443,
 260444,
 260896,
 260899,
 260903,
 262079,
 263614,
 263981,
 266066,
 266093,
 266924,
 267202,
 270616,
 271972,
 273804,
 276915,
 281191,
 282237,
 283010,
 290549,
 290550,
 290551,
 295195,
 298750,
 299191,
 30

In [19]:
import pandas as pd
import requests
import io
import time

 
# --- STEP B: FILTER FOR 'CONFIRMATORY' ASSAYS ONLY ---
print("Filtering for 'Confirmatory' assays...")
confirmatory_aids = []

for aid in all_aids:
    summary_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/assay/aid/{aid}/summary/JSON"
    res = requests.get(summary_url).json()
    
    try:
        # FIX: Changed from 'ActivityType' to 'Method' based on the raw JSON layout
        assay_type = res['AssaySummaries']['AssaySummary'][0]['Method']
        
        if assay_type.lower() == 'confirmatory':
            confirmatory_aids.append(aid)
            print(f" -> AID {aid} is Confirmatory. Added!")
        else:
            print(f" -> AID {aid} skipped (Method is '{assay_type}')")
            
    except KeyError:
        print(f" -> ⚠️ Warning: Unexpected JSON structure for AID {aid}")
        continue
        
    time.sleep(0.2) 

print(f"\nRetained {len(confirmatory_aids)} 'Confirmatory' assays.")
print(f"Active Confirmatory List: {confirmatory_aids}")


Filtering for 'Confirmatory' assays...
 -> AID 1433 skipped (Method is 'other')
 -> AID 1622 skipped (Method is 'screening')
 -> AID 1904 skipped (Method is 'other')
 -> AID 1982 skipped (Method is 'other')
 -> AID 3366 is Confirmatory. Added!
 -> ⚠️ Warning: Unexpected JSON structure for AID 3367
 -> AID 40020 is Confirmatory. Added!
 -> AID 40021 is Confirmatory. Added!
 -> AID 42142 is Confirmatory. Added!
 -> AID 51584 is Confirmatory. Added!
 -> AID 68095 is Confirmatory. Added!
 -> AID 68096 is Confirmatory. Added!
 -> AID 68097 is Confirmatory. Added!
 -> AID 68098 is Confirmatory. Added!
 -> AID 68099 is Confirmatory. Added!
 -> AID 68100 is Confirmatory. Added!
 -> AID 68101 is Confirmatory. Added!
 -> AID 68102 is Confirmatory. Added!
 -> AID 68103 is Confirmatory. Added!
 -> ⚠️ Warning: Unexpected JSON structure for AID 68104
 -> AID 68105 is Confirmatory. Added!
 -> AID 68106 is Confirmatory. Added!
 -> AID 69712 is Confirmatory. Added!
 -> AID 69713 is Confirmatory. Added!

In [22]:
confirmatory_aids

[3366,
 40020,
 40021,
 42142,
 51584,
 68095,
 68096,
 68097,
 68098,
 68099,
 68100,
 68101,
 68102,
 68103,
 68105,
 68106,
 69712,
 69713,
 69714,
 69715,
 69874,
 70241,
 79614,
 79617,
 81560,
 81567,
 81574,
 81579,
 81580,
 81581,
 81584,
 81723,
 81724,
 91107,
 91108,
 92132,
 105058,
 144357,
 164334,
 164335,
 164336,
 164339,
 164340,
 215071,
 218938,
 219061,
 226205,
 226585,
 235920,
 240497,
 240554,
 240635,
 240906,
 241491,
 242383,
 244158,
 248902,
 248914,
 254695,
 254834,
 254920,
 254948,
 254975,
 255842,
 255868,
 255958,
 256709,
 258046,
 259431,
 260896,
 260899,
 260903,
 263981,
 266066,
 266093,
 266924,
 267202,
 270616,
 271972,
 273804,
 276915,
 281191,
 282237,
 283010,
 290549,
 290550,
 295195,
 298750,
 304938,
 305799,
 306289,
 308241,
 308515,
 308828,
 309857,
 309859,
 310411,
 314705,
 314962,
 314965,
 315244,
 315250,
 320572,
 320574,
 325576,
 325824,
 328043,
 331423,
 331537,
 341698,
 344457,
 349121,
 361654,
 361656,
 364803,
 3

In [23]:
# --- STEP C: PULL DATA AND FILTER FOR 'PUBCHEM_ACTIVITY_OUTCOME' ---
print("Extracting compounds with active outcomes...")
all_compounds_list = []

if not confirmatory_aids:
    print("⚠️ Warning: 'confirmatory_aids' list is empty! Step B did not find matching assays.")
else:
    for aid in confirmatory_aids:
        csv_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/assay/aid/{aid}/csv"
        csv_res = requests.get(csv_url)
        
        if csv_res.status_code == 200:
            assay_df = pd.read_csv(io.StringIO(csv_res.text))
            
            # Find the outcome column dynamically (handling case sensitivity or spaces)
            outcome_col = [col for col in assay_df.columns if 'activity_outcome' in col.lower()]
            cid_col = [col for col in assay_df.columns if 'cid' in col.lower()]
            
            if outcome_col and cid_col:
                col_name = outcome_col[0]
                cid_name = cid_col[0]
                
                # Convert to string and handle both text ('active') and standard PubChem numerical mappings
                # In PubChem raw data: 2 often maps to 'Active'
                active_rows = assay_df[
                    assay_df[col_name].astype(str).str.lower().str.contains('active') | 
                    (assay_df[col_name].astype(str) == '2')
                ]
                
                if not active_rows.empty:
                    all_compounds_list.append(active_rows[[cid_name, col_name]].rename(columns={cid_name: 'PUBCHEM_CID', col_name: 'PUBCHEM_ACTIVITY_OUTCOME'}))
        time.sleep(0.2)


Extracting compounds with active outcomes...


In [ ]:

# --- STEP D: RETRIEVE CANONICAL SMILES & DROP DUPLICATES ---
# Defensive check: Only concatenate if we actually found data
if len(all_compounds_list) > 0:
    final_pool = pd.concat(all_compounds_list, ignore_index=True)

    # Drop duplicate IDs from the rows
    final_pool = final_pool.dropna(subset=['PUBCHEM_CID'])
    final_pool = final_pool.drop_duplicates(subset=['PUBCHEM_CID'])
    unique_cids = final_pool['PUBCHEM_CID'].astype(int).tolist()

    print(f"Total Unique Active CIDs captured: {len(unique_cids)}")

    # Fetch SMILES string formats for the structure screening
    smiles_data = []
    for i in range(0, len(unique_cids), 200):
        batch = unique_cids[i:i+200]
        cid_str = ",".join(map(str, batch))
        property_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid_str}/property/CanonicalSMILES/CSV"
        prop_res = requests.get(property_url)
        if prop_res.status_code == 200:
            smiles_data.append(pd.read_csv(io.StringIO(prop_res.text)))

    if smiles_data:
        df_smiles = pd.concat(smiles_data, ignore_index=True)
        df_smiles.to_csv("pubchem_erbb2_screening_set.csv", index=False)
        print(f"Done! Saved file with structures. Final shape match: {df_smiles.shape[0]} compounds.")
    else:
        print("❌ Error: Could not fetch SMILES properties from PubChem for the extracted CIDs.")
else:
    print("❌ Error: No compounds matched the 'Active' criteria across any of the assays. 'all_compounds_list' is empty.")

Total Unique Active CIDs captured: 4593
Done! Saved file with structures. Final shape match: 4593 compounds.


In [ ]:
# import pubchempy as pcp

# # Search for a compound by its common name
# compound_name = "Aspirin"
# results = pcp.get_compounds(compound_name, 'name')

# if results:
#     # Grab the first match found
#     compound = results[0]
    
#     print(f"Compound Name: {compound_name}")
#     print(f"PubChem CID:  {compound.cid}")
#     print(f"Formula:      {compound.molecular_formula}")
#     print(f"Mol. Weight:  {compound.molecular_weight} g/mol")
#     print(f"SMILES:       {compound.isomeric_smiles}")
#     print(f"IUPAC Name:   {compound.iupac_name}")
# else:
#     print("Compound not found.")
